In [2]:
!pip install duckdb


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import pandas as pd
import duckdb
from IPython.core.magic import register_cell_magic


@register_cell_magic
def sql(line, cell):
  df_var =  "df__" if line is None else line
  globals()[df_var] = duckdb.sql(f"""{cell}""").df()
  return globals()[df_var]

In [4]:
%%sql

select * from df

CatalogException: Catalog Error: Table with name df does not exist!
Did you mean "pg_depend"?

[Dados Abertos Voos e Operações Aereas](https://sistemas.anac.gov.br/dadosabertos/Voos%20e%20opera%C3%A7%C3%B5es%20a%C3%A9reas/Voo%20Regular%20Ativo%20%28VRA%29/2025/)

In [5]:
!wget "https://sistemas.anac.gov.br/dadosabertos/Voos%20e%20opera%C3%A7%C3%B5es%20a%C3%A9reas/Voo%20Regular%20Ativo%20%28VRA%29/2025/08%20-%20Agosto/VRA_20258.csv"
!wget "https://sistemas.anac.gov.br/dadosabertos/Voos%20e%20opera%C3%A7%C3%B5es%20a%C3%A9reas/Voo%20Regular%20Ativo%20%28VRA%29/2025/09%20-%20Setembro/VRA_20259.csv"
!wget "https://sistemas.anac.gov.br/dadosabertos/Voos%20e%20opera%C3%A7%C3%B5es%20a%C3%A9reas/Voo%20Regular%20Ativo%20%28VRA%29/2025/10%20-%20Outubro/VRA_202510.csv"
!wget "https://sistemas.anac.gov.br/dadosabertos/Voos%20e%20opera%C3%A7%C3%B5es%20a%C3%A9reas/Voo%20Regular%20Ativo%20%28VRA%29/2025/11%20-%20Novembro/VRA_202511.csv"

zsh:1: command not found: wget
zsh:1: command not found: wget
zsh:1: command not found: wget
zsh:1: command not found: wget


In [ ]:
VRA_20258 = pd.read_csv('VRA_20258.csv',header='infer',sep=';',skiprows=1, low_memory=False)
VRA_20259 = pd.read_csv('VRA_20259.csv',header='infer',sep=';',skiprows=1, low_memory=False)
VRA_202510 = pd.read_csv('VRA_202510.csv',header='infer',sep=';',skiprows=1, low_memory=False)
VRA_202511 = pd.read_csv('VRA_202511.csv',header='infer',sep=';',skiprows=1, low_memory=False)

VRA = pd.concat([VRA_20258, VRA_20259, VRA_202510, VRA_202511])


VRA = VRA.rename(columns={'ICAO Empresa Aérea': 'icao_empresa_aerea',
                          'Número Voo': 'numero_voo',
                          'Código Autorização (DI)': 'codigo_autorizacao',
                          'Código Tipo Linha': 'codigo_tipo_linha',
                          'ICAO Aeródromo Origem': 'icao_aerodromo_origem',
                          'ICAO Aeródromo Destino': 'icaco_aerodromo_destino',
                          'Partida Prevista': 'partica_prevista',
                          'Partida Real': 'partida_real',
                          'Chegada Prevista': 'chegada_prevista',
                          'Chegada Real': 'chegada_real',
                          'Situação Voo': 'situacao_voo',
                          'Código Justificativa': 'codigo_justificativa',
                          'Justificativa': 'justificativa',
                          'Código Resposta Autorização': 'codigo_resposta_autorizacao',
                          'Resposta Autorização': 'resposta_autorizacao',
                          })

VRA['icao_empresa_aerea'] = VRA['icao_empresa_aerea'].astype(str)
VRA['partida_real'] = pd.to_datetime(VRA['partida_real'], errors='coerce')

VRA['year'] = VRA['partida_real'].dt.year
VRA['month'] = VRA['partida_real'].dt.month
VRA['day'] = VRA['partida_real'].dt.day

VRA['year'] = VRA['year'].fillna(0).astype(int)
VRA['month'] = VRA['month'].fillna(0).astype(int)
VRA['day'] = VRA['day'].fillna(0).astype(int)

VRA.dtypes

In [ ]:
VRA.to_csv('VRA_full.csv', index=False)

In [ ]:
!ls -lh

In [ ]:
VRA.to_parquet('VRA_full.parquet', index=False,compression='snappy')

In [ ]:
VRA.to_parquet('.', index=False,compression='snappy', partition_cols=['year','month'])

In [ ]:
!ls -lhR year\=2025

'year=2025':
total 24K
drwxr-xr-x 2 root root 4.0K Jan  8 04:13 'month=10'
drwxr-xr-x 2 root root 4.0K Jan  8 04:13 'month=11'
drwxr-xr-x 2 root root 4.0K Jan  8 04:13 'month=12'
drwxr-xr-x 2 root root 4.0K Jan  8 04:13 'month=7'
drwxr-xr-x 2 root root 4.0K Jan  8 04:13 'month=8'
drwxr-xr-x 2 root root 4.0K Jan  8 04:13 'month=9'

'year=2025/month=10':
total 1.9M
-rw-r--r-- 1 root root 1.9M Jan  8 04:13 699ff0dcd8a34e9fbe224efff4aadb9f-0.parquet

'year=2025/month=11':
total 1.8M
-rw-r--r-- 1 root root 1.8M Jan  8 04:13 699ff0dcd8a34e9fbe224efff4aadb9f-0.parquet

'year=2025/month=12':
total 20K
-rw-r--r-- 1 root root 18K Jan  8 04:13 699ff0dcd8a34e9fbe224efff4aadb9f-0.parquet

'year=2025/month=7':
total 12K
-rw-r--r-- 1 root root 11K Jan  8 04:13 699ff0dcd8a34e9fbe224efff4aadb9f-0.parquet

'year=2025/month=8':
total 1.8M
-rw-r--r-- 1 root root 1.8M Jan  8 04:13 699ff0dcd8a34e9fbe224efff4aadb9f-0.parquet

'year=2025/month=9':
total 1.8M
-rw-r--r-- 1 root root 1.8M Jan  8 04:13 699ff0dcd8

In [ ]:
%%sql
select * from VRA

,icao_empresa_aerea,numero_voo,codigo_autorizacao,codigo_tipo_linha,icao_aerodromo_origem,icaco_aerodromo_destino,partica_prevista,partida_real,chegada_prevista,chegada_real,situacao_voo,codigo_justificativa,year,month,day
0,LAN,0715,0,I,SBGR,SCEL,2025-08-24 07:10:00,2025-08-24 07:05:00,2025-08-24 11:30:00,2025-08-24 11:19:00,REALIZADO,NaN,2025,8,24
1,LAN,0715,0,I,SBGR,SCEL,2025-08-25 07:10:00,2025-08-25 07:14:00,2025-08-25 11:30:00,2025-08-25 11:16:00,REALIZADO,NaN,2025,8,25
2,LAN,0715,0,I,SBGR,SCEL,2025-08-26 07:10:00,2025-08-26 07:17:00,2025-08-26 11:30:00,2025-08-26 11:03:00,REALIZADO,NaN,2025,8,26
3,LAN,0715,0,I,SBGR,SCEL,2025-08-27 07:10:00,2025-08-27 07:08:00,2025-08-27 11:30:00,2025-08-27 10:56:00,REALIZADO,NaN,2025,8,27
4,LAN,0715,0,I,SBGR,SCEL,2025-08-28 07:10:00,2025-08-28 07:12:00,2025-08-28 11:30:00,2025-08-28 11:15:00,REALIZADO,NaN,2025,8,28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
335050,TAM,8035,0,I,SABE,SBGR,2025-11-16 18:20:00,2025-11-16 18:11:00,2025-11-16 21:05:00,2025-11-16 20:55:00,REALIZADO,NaN,2025,11,16
335051,TAM,8035,0,I,SABE,SBGR,2025-11-17 18:20:00,2025-11-17 18:11:00,2025-11-17 21:05:00,2025-11-17 20:48:00,REALIZADO,NaN,2025,11,17
335052,TAM,8035,0,I,SABE,SBGR,2025-11-18 18:20:00,2025-11-18 18:32:00,2025-11-18 21:05:00,2025-11-18 21:09:00,REALIZADO,NaN,2025,11,18
335053,TAM,8035,0,I,SABE,SBGR,2025-11-19 18:20:00,2025-11-19 18:46:00,2025-11-19 21:05:00,2025-11-19 21:18:00,REALIZADO,NaN,2025,11,19
